###  1. Imports + env

In [2]:
import os
import json
import urllib.request
import urllib.error
import urllib.parse
from datetime import datetime, timedelta
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import ipywidgets as widgets
from IPython.display import display
from dotenv import load_dotenv

plt.style.use("default")
%matplotlib inline

load_dotenv()
API_KEY = os.getenv("VISUAL_CROSSING_KEY")
LIVE_LOCATION = os.getenv("VISUAL_CROSSING_LOCATION")
if not API_KEY:
    raise ValueError("VISUAL_CROSSING_KEY not found in .env")
if not LIVE_LOCATION:
    raise ValueError("VISUAL_CROSSING_LOCATION not found in .env")


PermissionError: [Errno 1] Operation not permitted

### 2. Load Processed Data + Trained Models
What this cell does:
- Loads the cleaned hourly weather dataset.
- Loads each trained horizon model and its metadata.

Why this step exists:
- The dashboard needs historical data and matching model feature schemas before any prediction or plotting.


In [12]:
DATA_PATH = Path("../data/processed/weather_hourly_clean.csv")
MODELS_DIR = Path("../models")

df = pd.read_csv(DATA_PATH, parse_dates=["datetime"]).sort_values("datetime").reset_index(drop=True)

models = {}
meta_info = {}
for name in ["B_next_1h", "C_next_3h", "D_next_6h"]:
    models[name] = joblib.load(MODELS_DIR / f"hgb_{name}.pkl")
    meta_info[name] = json.loads((MODELS_DIR / f"hgb_{name}_meta.json").read_text())

print("Models loaded:", list(models.keys()))


Models loaded: ['B_next_1h', 'C_next_3h', 'D_next_6h']


### 3. Create Dashboard Controls
What this cell does:
- Defines all interactive controls: horizon, threshold, date range, live field, live overlay, refresh button.
- Defines schema inspector widgets for live columns, model features, and missing features.

Why this step exists:
- Interactive controls let you inspect model behavior and live-data compatibility in a reproducible way.


In [1]:
LIVE_API_FIELDS = [
    "temp", "humidity", "pressure", "cloudcover", "windspeed", "visibility", "precip",
    "feelslike", "dew", "precipprob", "snow", "snowdepth", "windgust", "winddir",
    "sealevelpressure", "solarradiation", "solarenergy", "uvindex", "severerisk"
]

date_slider = widgets.SelectionRangeSlider(
    options=df["datetime"],
    index=(len(df)-1000, len(df)-1),
    description="Date range",
    layout={'width': '800px'}
)

horizon_dropdown = widgets.Dropdown(
    options=["B_next_1h", "C_next_3h", "D_next_6h"],
    value="D_next_6h",
    description="Horizon"
)

threshold_slider = widgets.FloatSlider(
    value=0.05,
    min=0.01,
    max=0.5,
    step=0.01,
    description="Threshold"
)

live_feature_dropdown = widgets.Dropdown(
    options=LIVE_API_FIELDS,
    value="temp",
    description="Live field"
)

live_overlay_toggle = widgets.Checkbox(value=True, description="Overlay LIVE point")

refresh_button = widgets.Button(description="Refresh Live Data", button_style="success")
live_status = widgets.HTML(value="<b>Live:</b> not fetched yet")
live_feature_status = widgets.HTML(value="<b>Live field value:</b> n/a")

schema_live_cols = widgets.SelectMultiple(
    options=[],
    value=[],
    description="Live cols",
    layout={"width": "800px", "height": "140px"},
    disabled=True
)
schema_model_feats = widgets.SelectMultiple(
    options=[],
    value=[],
    description="Model feats",
    layout={"width": "800px", "height": "140px"},
    disabled=True
)
schema_missing = widgets.SelectMultiple(
    options=[],
    value=[],
    description="Missing",
    layout={"width": "800px", "height": "100px"},
    disabled=True
)

schema_box = widgets.VBox([
    widgets.HTML("<b>Schema Inspector (Live API vs Model Features)</b>"),
    schema_live_cols,
    schema_model_feats,
    schema_missing
])

controls = widgets.VBox([
    horizon_dropdown,
    threshold_slider,
    date_slider,
    live_feature_dropdown,
    live_overlay_toggle,
    widgets.HBox([refresh_button, live_status]),
    live_feature_status,
    schema_box
])




NameError: name 'widgets' is not defined

### 4. Define Live API Fetch + Parsing Helpers
What this cell does:
- Builds the Visual Crossing request URL.
- Fetches JSON live data and converts it into a numeric hourly DataFrame.

Why this step exists:
- Live overlay and schema checks require a clean, structured live input table.


In [14]:
BASE_URL = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/"
LOCATION = LIVE_LOCATION
UNIT_GROUP = "metric"
CONTENT_TYPE = "json"

def fetch_live_vc_json():
    end = datetime.utcnow().date()
    start = end - timedelta(days=2)
    include = "hours,current"

    request_url = (
        f"{BASE_URL}"
        f"{urllib.parse.quote_plus(LOCATION)}/"
        f"{start.isoformat()}/{end.isoformat()}"
        f"?unitGroup={UNIT_GROUP}"
        f"&include={include}"
        f"&contentType={CONTENT_TYPE}"
        f"&key={API_KEY}"
    )

    try:
        with urllib.request.urlopen(request_url, timeout=60) as response:
            data = response.read()
        return json.loads(data)

    except urllib.error.HTTPError as e:
        raise RuntimeError(f"HTTP Error {e.code}: {e.read().decode()}")

    except urllib.error.URLError as e:
        raise RuntimeError(f"URL Error: {e.reason}")

    except json.JSONDecodeError:
        raise RuntimeError("Error decoding JSON response.")


def vc_json_to_df(weather_json, hours_back=48):
    records = []
    for day in weather_json.get("days", []):
        for h in day.get("hours", []):
            dt = pd.to_datetime(h.get("datetime"))
            records.append({
                "datetime": dt,

                "temp": h.get("temp"),
                "humidity": h.get("humidity"),
                "pressure": h.get("pressure"),
                "cloudcover": h.get("cloudcover"),
                "windspeed": h.get("windspeed"),
                "visibility": h.get("visibility"),
                "precip": h.get("precip", 0.0),

                "feelslike": h.get("feelslike"),
                "dew": h.get("dew"),
                "precipprob": h.get("precipprob"),
                "snow": h.get("snow"),
                "snowdepth": h.get("snowdepth"),
                "windgust": h.get("windgust"),
                "winddir": h.get("winddir"),
                "sealevelpressure": h.get("sealevelpressure"),
                "solarradiation": h.get("solarradiation"),
                "solarenergy": h.get("solarenergy"),
                "uvindex": h.get("uvindex"),
                "severerisk": h.get("severerisk"),
            })

    df_live = pd.DataFrame(records).sort_values("datetime")
    for c in df_live.columns:
        if c != "datetime":
            df_live[c] = pd.to_numeric(df_live[c], errors="coerce")
    return df_live.tail(hours_back).reset_index(drop=True)


### 5. Add Live Refresh Logic
What this cell does:
- Stores fetched live data in a cache.
- Updates status text and selected live-field value after refresh.

Why this step exists:
- Decouples live data retrieval from plotting so dashboard updates remain fast and controlled.


In [15]:
LIVE_CACHE = {"df_live": None, "fetched_at": None}

def update_live_feature_status(_=None):
    if LIVE_CACHE["df_live"] is None or LIVE_CACHE["df_live"].empty:
        live_feature_status.value = "<b>Live field value:</b> n/a"
        return

    selected = live_feature_dropdown.value
    latest = LIVE_CACHE["df_live"].iloc[-1]

    if selected in LIVE_CACHE["df_live"].columns:
        value = latest[selected]
        live_feature_status.value = (
            f"<b>Live field value:</b> {selected} = {value} at {latest['datetime']}"
        )
    else:
        live_feature_status.value = f"<b>Live field value:</b> {selected} not available in live payload"


def refresh_live(_=None):
    try:
        wj = fetch_live_vc_json()
        df_live = vc_json_to_df(wj, hours_back=48)

        LIVE_CACHE["df_live"] = df_live
        LIVE_CACHE["fetched_at"] = datetime.utcnow()

        available = [c for c in LIVE_API_FIELDS if c in df_live.columns]
        if available:
            live_feature_dropdown.options = available
            if live_feature_dropdown.value not in available:
                live_feature_dropdown.value = available[0]

        live_status.value = f"<b>Live:</b> fetched at {LIVE_CACHE['fetched_at'].strftime('%Y-%m-%d %H:%M:%S')} UTC"
        update_live_feature_status()

    except Exception as e:
        live_status.value = f"<b>Live:</b> error: {e}"

refresh_button.on_click(refresh_live)
live_feature_dropdown.observe(update_live_feature_status, names="value")




### 6. Build Historical Features Per Horizon
What this cell does:
- Creates the target (rain in next 1h/3h/6h) using model metadata.
- Returns feature matrix and labels for historical evaluation.

Why this step exists:
- Keeps training-time feature/target logic aligned with dashboard evaluation.


In [16]:
def build_features(df_in, horizon_name):
    meta = meta_info[horizon_name]
    h = meta["hours"]
    FEATURES = meta["features"]

    df_temp = df_in.copy()

    # target: any rain in next h hours (h=1,3,6); for same-hour model you'd handle separately
    df_temp["target"] = pd.concat(
        [df_temp["rain_1h"].shift(-i) for i in range(1, h + 1)],
        axis=1
    ).max(axis=1)

    df_temp = df_temp.dropna(subset=["target"]).copy()
    df_temp["target"] = df_temp["target"].astype(int)

    X = df_temp[FEATURES].copy()
    y = df_temp["target"].copy()
    return df_temp, X, y


### 7. Build Live Features + Schema Inspector
What this cell does:
- Creates any rolling live features needed by the model.
- Compares live columns against model features and lists missing columns.
- Produces a single-row live feature vector for prediction.

Why this step exists:
- Prevents silent schema mismatch and makes live prediction inputs transparent.


In [17]:
def build_live_features(df_live, horizon_name):
    meta = meta_info[horizon_name]
    FEATURES = meta["features"]

    df_live = df_live.copy()

    # rolling features used in training (if expected)
    if "precip" in df_live.columns:
        df_live["rain_6h"] = df_live["precip"].rolling(6, min_periods=1).sum()
        df_live["rain_24h"] = df_live["precip"].rolling(24, min_periods=1).sum()

    # schema inspector
    live_cols = [c for c in df_live.columns if c != "datetime"]
    model_feats = list(FEATURES)
    missing = [f for f in model_feats if f not in df_live.columns]

    schema_live_cols.options = live_cols
    schema_model_feats.options = model_feats
    schema_missing.options = missing if missing else ["(none)"]

    # create missing cols as NaN so we can form X_live
    for f in missing:
        df_live[f] = np.nan

    latest = df_live.iloc[-1]
    X_live = latest[FEATURES].to_frame().T

    # future-proof filling (no deprecated fillna(method=...))
    X_live = X_live.apply(pd.to_numeric, errors="coerce")
    X_live = X_live.ffill(axis=1).bfill(axis=1).fillna(0.0)
    X_live = X_live.infer_objects(copy=False)

    return X_live, latest, missing


### 8. Define Dashboard Update Function
What this cell does:
- Computes historical probabilities and thresholded predictions.
- Plots probability timeline and optional live-point overlay.
- Prints precision/recall/F1 and selected live-field value.

Why this step exists:
- This is the main analysis view for comparing model output, threshold behavior, and live context.


In [18]:
def update_dashboard(horizon_name, threshold, date_range, live_feature, live_overlay=True):
    # Historical
    df_temp, X, y = build_features(df, horizon_name)
    model = models[horizon_name]

    df_temp["prob"] = model.predict_proba(X)[:, 1]
    df_temp["pred"] = (df_temp["prob"] >= threshold).astype(int)

    mask = (df_temp["datetime"] >= date_range[0]) & (df_temp["datetime"] <= date_range[1])
    df_plot = df_temp.loc[mask].copy()

    plt.figure(figsize=(14, 5))
    plt.plot(df_plot["datetime"], df_plot["prob"], label="Probability")
    plt.axhline(threshold, color="red", linestyle="--", label="Threshold")

    plt.scatter(
        df_plot.loc[df_plot["target"] == 1, "datetime"],
        df_plot.loc[df_plot["target"] == 1, "prob"],
        color="black", s=15, label="Actual rain"
    )

    # Live overlay from cache
    if live_overlay and LIVE_CACHE["df_live"] is not None:
        X_live, latest, missing = build_live_features(LIVE_CACHE["df_live"], horizon_name)
        live_prob = model.predict_proba(X_live)[0, 1]
        live_ts = latest["datetime"]
        plt.scatter([live_ts], [live_prob], s=90, marker="o", label="LIVE now")

    plt.legend()
    plt.title(f"{horizon_name} Rain Forecast")
    plt.show()

    # Metrics
    from sklearn.metrics import precision_score, recall_score, f1_score
    precision = precision_score(df_plot["target"], df_plot["pred"], zero_division=0)
    recall = recall_score(df_plot["target"], df_plot["pred"], zero_division=0)
    f1 = f1_score(df_plot["target"], df_plot["pred"], zero_division=0)

    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"F1:        {f1:.3f}")

    if live_overlay:
        if LIVE_CACHE["df_live"] is None:
            print("\n(Live overlay) Click Refresh Live Data first.")
        else:
            print(f"\n(Live overlay) Uses cached live fetched at: {LIVE_CACHE['fetched_at']} UTC")

    if LIVE_CACHE["df_live"] is None:
        print(f"(Live field) {live_feature}: n/a (refresh live data first)")
    else:
        latest = LIVE_CACHE["df_live"].iloc[-1]
        if live_feature in LIVE_CACHE["df_live"].columns:
            print(f"(Live field) {live_feature}: {latest[live_feature]} at {latest['datetime']}")
        else:
            print(f"(Live field) {live_feature}: not available in live payload")




### 9. Bind Widgets to the Dashboard Output
What this cell does:
- Connects widget state to the update function.
- Displays controls and live plot/output together.

Why this step exists:
- Activates the interactive dashboard for iterative model inspection.


In [19]:
out = widgets.interactive_output(
    update_dashboard,
    {
        "horizon_name": horizon_dropdown,
        "threshold": threshold_slider,
        "date_range": date_slider,
        "live_feature": live_feature_dropdown,
        "live_overlay": live_overlay_toggle,
    },
)

display(widgets.VBox([controls, out]))


